# CO-SPY pretrained — Eval FLUX.1-schnell

Eval **5.000 real MSCOCO val2017 + 5.000 fake FLUX.1-schnell** bằng **CO-SPY pretrained Fusion v1**.

**Output:** AP, Accuracy, thời gian chạy, `result.json`, `predictions.csv`.

> Kaggle: bật **GPU** và **Internet**.

In [ ]:
from pathlib import Path
import os, sys, subprocess, time, json, csv

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "cospy"
BATCH = 16

# Clone source
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/datnguyentrung/CO-SPY.git", str(REPO)],
        check=True
    )

# Dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "open-clip-torch==2.31.0",
    "diffusers==0.32.2",
    "transformers==4.46.3",
    "accelerate==1.0.1",
    "safetensors==0.5.3",
    "ftfy==6.2.3",
    "timm==1.0.15",
    "loguru==0.7.3",
], check=True)

print("Setup ready")

In [ ]:
# Weights
PRETRAIN = Path(
    "/kaggle/input/notebooks/datnguyentrung/dl-pretrained/"
    "sd-v1_4/sd-v1_4"
)
DST = REPO / "pretrained" / "sd-v1_4"
DST.mkdir(parents=True, exist_ok=True)

for name in [
    "artifact_weights.pth",
    "semantic_weights.pth",
    "fusion_weights.pth",
]:
    src = PRETRAIN / name
    dst = DST / name
    assert src.is_file(), f"Missing: {src}"
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src)

# Real: MSCOCO val2017
REAL_DIR = Path(
    "/kaggle/input/datasets/datnguyentrung/co-spy-v1/"
    "mscoco2017/mscoco2017/val2017"
)

# Fake: notebook output FLUX-schnell
NOTEBOOK_ROOT = Path("/kaggle/input/notebooks/datnguyentrung")
FLUX_ROOTS = [
    p for p in NOTEBOOK_ROOT.iterdir()
    if p.is_dir() and "flux" in p.name.lower()
]
assert FLUX_ROOTS, "Không tìm thấy input FLUX-schnell"
FLUX_DIR = FLUX_ROOTS[0]

EXT = {".jpg", ".jpeg", ".png", ".webp"}

real_images = sorted(
    p for p in REAL_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in EXT
)[:5000]

fake_images = sorted(
    p for p in FLUX_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in EXT
)[:5000]

assert len(real_images) == 5000, f"Real: {len(real_images)}/5000"
assert len(fake_images) == 5000, f"Fake: {len(fake_images)}/5000"

print(f"Ready: {len(real_images)} real + {len(fake_images)} fake")

In [ ]:
# Load pretrained Fusion v1
import torch

os.chdir(REPO)
sys.path.insert(0, str(REPO))

from evaluate import Detector

device = "cuda:0" if torch.cuda.is_available() else "cpu"
assert device == "cuda:0", "Hãy bật GPU trên Kaggle"

model = Detector(
    device=device,
    mode="fusion",
    train_dataset="sd-v1_4",
    pretrain=True,
    fusion_head="v1",
).model

model.eval()
print("GPU:", torch.cuda.get_device_name(0))
print("Model ready")

In [ ]:
# Eval + save
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, accuracy_score

class EvalDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = model.test_transform(im.convert("RGB"))
        return x, self.labels[i]

paths = real_images + fake_images
labels = [0] * 5000 + [1] * 5000

loader = DataLoader(
    EvalDataset(paths, labels),
    batch_size=BATCH,
    shuffle=False,
    num_workers=0,
)

scores, y_true = [], []
start = time.time()

with torch.inference_mode():
    for images, labels_batch in tqdm(
        loader, desc="CO-SPY eval", unit="batch"
    ):
        scores.extend(float(x) for x in model.predict(images))
        y_true.extend(labels_batch.tolist())

elapsed = time.time() - start
scores = np.asarray(scores)
y_true = np.asarray(y_true)
y_pred = (scores >= 0.5).astype(int)

ap = average_precision_score(y_true, scores)
acc = accuracy_score(y_true, y_pred)
real_acc = (scores[y_true == 0] < 0.5).mean()
fake_acc = (scores[y_true == 1] >= 0.5).mean()

result = {
    "AP": float(ap),
    "Accuracy": float(acc),
    "Real_correct": float(real_acc),
    "Fake_detected": float(fake_acc),
    "Runtime_minutes": float(elapsed / 60),
}

print("\n" + "=" * 52)
print(f"AP                 : {ap * 100:.2f}%")
print(f"Accuracy           : {acc * 100:.2f}%")
print(f"Real correctly real: {real_acc * 100:.2f}%")
print(f"Fake detected AI   : {fake_acc * 100:.2f}%")
print(f"Runtime            : {elapsed / 60:.1f} min")
print("=" * 52)

OUT = WORK / "cospy_flux_schnell_eval"
OUT.mkdir(exist_ok=True)

with open(OUT / "result.json", "w") as f:
    json.dump(result, f, indent=2)

with open(OUT / "predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["path", "label", "score", "predicted_label"])
    for p, y, s, pred in zip(paths, y_true, scores, y_pred):
        w.writerow([str(p), int(y), float(s), int(pred)])

print("\nSaved:", OUT)